# Running Pre-Trained CGNet Antarctic Model

Purpose:
--------
The purpose of this notebook is to run pre-trained CGnet Arctic models for machine learning detection of atmospheric rivers and tropical cyclones.\
See ClimateNet repo here: https://github.com/andregraubner/ClimateNet

Authors/Contributors:
---------------------
* Teagan King
* John Truesdale
* Katie Dagon

## Import libraries

In [1]:
import os
import sys
import numpy as np

sys.path.append("/glade/work/tking/cgnet/ClimateNet") # append path to ClimateNet repo
from climatenet.utils.data import ClimateDatasetLabeled, ClimateDataset
from climatenet.models import CGNet
from climatenet.utils.utils import Config
from climatenet.track_events import track_events
from climatenet.analyze_events import analyze_events
from climatenet.visualize_events import visualize_events

from os import path

## Confirm GPU resources
Can request through JupyterHub launch page.\
Current resources request (2/15/23): 1 node, 4 cpu, 64GB mem, 2 V100 GPU\
4/5/23: Noticed that not all CPU/mem was needed, new request: 1 node, 2 cpu, 64GB mem, 2 V100 GPU\
4/6/23: Trying out an increase in GPU to see if that improves speed: 1 node, 2 cpu, 64GB mem, 3 V100 GPU

In [2]:
# requires loading pytorch into environment
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())

True
2


## Function for creating TC/AR masks using pre-trained model

In [3]:
def cgnet_load_create_masks(model_path, inference_path, save_dir, analyze=False, visualize=False):
    """Use a pre-trained model to create masks of tropical cyclones (mask value = 1)
    and atmospheric rivers (mask value = 2)

    Function will create NetCDF mask files and save them in the save_directory.

    Parameters:
    -----------
    model_path: str
        filepath to pre-trained CGnet model
    inference_path: str
        filepath to inference data
    save_dir: str
        filepath to where the masks will be saved as .nc files
    analyze: bool (optional)
        default is False; if True, will save plots for analyzing events using climatenet.analyze_events().
        Note that this can significantly increase the time to run.
    visualize : bool (optional)
        default is False; if True, will save plots for visualizing events using climatenet.visualize_events().
        Note that this can significantly increase the time to run.

    """
    # instantiate CGNet with pre-trained model
    cgnet = CGNet(model_path=model_path)

    # inference using the pre-trained config file
    inference = ClimateDataset(inference_path, cgnet.config)
    class_masks = cgnet.predict(inference)

    # create save dir, if needed
    if not os.path.isdir(save_dir):
        os.makedirs(save_dir)
    else:
        print("Warning: might overwrite {}".format(save_dir))

    # save out class masks
    class_masks.name = 'masks'
    class_masks.to_netcdf(save_dir+"/class_masks.nc")
    print("Saved class masks to {}".format(save_dir))

    # note: this is resource intensive
    # event_masks = track_events(class_masks) # masks with event IDs
    # event_masks.to_netcdf(save_dir+"/event_masks.nc")
    # print("Saved event masks to {}".format(save_dir))

    if analyze:
        analyze_events(event_masks, class_masks, save_dir+"/")
        print("Analyze events done")
    if visualize:
        visualize_events(event_masks, inference, save_dir+"/")
        print("Visualize events done")

    return

# Pre-trained model which uses TMQ

## Load pre-trained model
No need to specify config, just the folder with config/weights will work

In [3]:
cgnet = CGNet(model_path='/glade/work/tking/cgnet/ML-extremes/trained_models/trained_cgnet.062024_TMQ')

In [4]:
cgnet

In [5]:
cgnet.config

### CESM Historical output, 2000-2005
2/23/23\
Memory use peaking at ~38GB during inference on each year\
Each year takes ~7 min to run

In [7]:
%%time
model_path = '/glade/work/tking/cgnet/ML-extremes/trained_models/trained_cgnet.062024_TMQ'
inference_dir = '/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/'

for year in [2002, 2005]: #2000, 2001, 2003, 2004
    inference_path = inference_dir+str(year)
    save_dir = inference_path+'/masks'
    cgnet_load_create_masks(model_path, inference_path, save_dir, analyze=False, visualize=False)

  0%|          | 0/730 [00:00<?, ?it/s]

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/2002/masks


100%|██████████| 730/730 [26:28<00:00,  2.18s/it]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/2005/masks
CPU times: user 6min 12s, sys: 2min 2s, total: 8min 15s
Wall time: 56min 47s


### CESM RCP2.6 output, 2006-2015
2/27/23: Each year takes ~9 min to run

In [7]:
%%time
model_path = '/glade/work/tking/cgnet/ML-extremes/trained_models/trained_cgnet.062024_TMQ'
inference_dir = '/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/'

for year in range(2015, 2016):
    inference_path = inference_dir+str(year)
    save_dir = inference_path+'/masks'
    cgnet_load_create_masks(model_path, inference_path, save_dir, analyze=False, visualize=False)

100%|██████████| 730/730 [23:22<00:00,  1.92s/it]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/2015/masks
CPU times: user 3min 3s, sys: 1min 4s, total: 4min 8s
Wall time: 23min 43s


### CESM RCP8.5 output, 2086-2100
2/27/23 new GPU job: Each year takes ~25 min to run

In [9]:
%%time
model_path = '/glade/work/tking/cgnet/ML-extremes/trained_models/trained_cgnet.062024_TMQ'
inference_dir = '/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/'

for year in range(2086, 2101):
    inference_path = inference_dir+str(year)
    save_dir = inference_path+'/masks'
    cgnet_load_create_masks(model_path, inference_path, save_dir, analyze=False, visualize=False)

100%|██████████| 730/730 [24:50<00:00,  2.04s/it]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2098/masks


100%|██████████| 730/730 [25:24<00:00,  2.09s/it]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2099/masks


100%|██████████| 730/730 [24:14<00:00,  1.99s/it]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2100/masks
CPU times: user 9min 21s, sys: 4min 26s, total: 13min 48s
Wall time: 1h 22min 33s


# Pre-trained model which uses PSL

In [4]:
cgnet = CGNet(model_path='/glade/work/tking/cgnet/ML-extremes/trained_models/trained_cgnet.092025_PSL')

In [5]:
inference = ClimateDataset('/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN', cgnet.config)

In [6]:
inference.fields

{'psl': {'mean': 101001.370879, 'std': 1387.300643}}

### CESM Historical output, 2000-2005
2/23/23\
Memory use peaking at ~38GB during inference on each year\
Each year takes ~7 min to run

In [12]:
%%time
model_path = '/glade/work/tking/cgnet/ML-extremes/trained_models/trained_cgnet.092025_PSL'
inference_dir = '/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/'

for year in range(2000, 2006):
    inference_path = inference_dir+str(year)
    save_dir = inference_path+'/masks_psl'
    cgnet_load_create_masks(model_path, inference_path, save_dir, analyze=False, visualize=False)

  0%|          | 0/730 [00:00<?, ?it/s]

/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/2000
/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/2000/masks_psl


100%|██████████| 730/730 [23:03<00:00,  1.90s/it]


  0%|          | 0/730 [00:00<?, ?it/s]

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/2000/masks_psl
/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/2001
/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/2001/masks_psl


...

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/2003/masks_psl
/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/2004
/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/2004/masks_psl


  0%|          | 0/730 [00:00<?, ?it/s]

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/2004/masks_psl
/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/2005
/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/2005/masks_psl


100%|██████████| 730/730 [21:45<00:00,  1.79s/it]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/2005/masks_psl
CPU times: user 18min 21s, sys: 6min 44s, total: 25min 6s
Wall time: 2h 16min 43s


### CESM RCP2.6 output, 2006-2015
2/27/23: Each year takes ~25 min to run

In [7]:
%%time
model_path = '/glade/work/tking/cgnet/ML-extremes/trained_models/trained_cgnet.092025_PSL'
inference_dir = '/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/'

for year in range(2011, 2015):
    inference_path = inference_dir+str(year)
    save_dir = inference_path+'/masks_psl'
    cgnet_load_create_masks(model_path, inference_path, save_dir, analyze=False, visualize=False)

  0%|          | 0/730 [00:00<?, ?it/s]

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/2011/masks_psl


100%|██████████| 730/730 [24:12<00:00,  1.99s/it]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/2012/masks_psl


  0%|          | 0/730 [00:00<?, ?it/s]

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/2013/masks_psl


100%|██████████| 730/730 [26:03<00:00,  2.14s/it]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/2014/masks_psl
CPU times: user 12min 9s, sys: 4min 27s, total: 16min 36s
Wall time: 1h 43min 58s


### CESM RCP8.5 output, 2086-2100
2/27/23 new GPU job: Each year takes ~25 min to run

In [8]:
%%time
model_path = '/glade/work/tking/cgnet/ML-extremes/trained_models/trained_cgnet.092025_PSL'
inference_dir = '/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/'

for year in range(2093, 2101):
    inference_path = inference_dir+str(year)
    save_dir = inference_path+'/masks_psl'
    cgnet_load_create_masks(model_path, inference_path, save_dir, analyze=False, visualize=False)

100%|██████████| 730/730 [25:00<00:00,  2.06s/it]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2093/masks_psl


100%|██████████| 730/730 [27:35<00:00,  2.27s/it]  


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2094/masks_psl


100%|██████████| 730/730 [26:08<00:00,  2.15s/it]  


...

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2098/masks_psl


100%|██████████| 730/730 [24:29<00:00,  2.01s/it]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2099/masks_psl


100%|██████████| 730/730 [25:07<00:00,  2.07s/it]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2100/masks_psl
CPU times: user 24min 38s, sys: 13min 35s, total: 38min 14s
Wall time: 3h 55min 44s


# Pre-trained model which uses TMQ and PSL

In [4]:
cgnet = CGNet(model_path='/glade/work/tking/cgnet/ML-extremes/trained_models/trained_cgnet.102025_PSL_TMQ')

### CESM Historical output, 2000-2005
2/23/23\
Memory use peaking at ~38GB during inference on each year\
Each year takes ~7 min to run

In [7]:
%%time
model_path = '/glade/work/tking/cgnet/ML-extremes/trained_models/trained_cgnet.102025_PSL_TMQ'
inference_dir = '/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/'

for year in range(2005, 2006):
    inference_path = inference_dir+str(year)
    save_dir = inference_path+'/masks_psl_tmq'
    cgnet_load_create_masks(model_path, inference_path, save_dir, analyze=False, visualize=False)

100%|██████████| 730/730 [21:27<00:00,  1.76s/it]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/B20TRC5CN/2005/masks_psl_tmq
CPU times: user 3min 10s, sys: 1min 11s, total: 4min 22s
Wall time: 21min 49s


### CESM RCP2.6 output, 2006-2015
2/27/23: Each year takes ~25 min to run

In [8]:
%%time
model_path = '/glade/work/tking/cgnet/ML-extremes/trained_models/trained_cgnet.102025_PSL_TMQ'
inference_dir = '/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/'

for year in range(2006, 2016):
    inference_path = inference_dir+str(year)
    save_dir = inference_path+'/masks_psl_tmq'
    cgnet_load_create_masks(model_path, inference_path, save_dir, analyze=False, visualize=False)

100%|██████████| 730/730 [24:17<00:00,  2.00s/it]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/2006/masks_psl_tmq


100%|██████████| 730/730 [23:08<00:00,  1.90s/it]


  0%|          | 0/730 [00:00<?, ?it/s]

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/2007/masks_psl_tmq


...

100%|██████████| 730/730 [25:17<00:00,  2.08s/it]


  0%|          | 0/730 [00:00<?, ?it/s]

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/2014/masks_psl_tmq


100%|██████████| 730/730 [25:18<00:00,  2.08s/it]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP26C5CN/2015/masks_psl_tmq
CPU times: user 33min 22s, sys: 12min 12s, total: 45min 35s
Wall time: 4h 12min 16s


### CESM RCP8.5 output, 2086-2100
2/27/23 new GPU job: Each year takes ~25 min to run

In [5]:
%%time
model_path = '/glade/work/tking/cgnet/ML-extremes/trained_models/trained_cgnet.102025_PSL_TMQ'
inference_dir = '/glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/'

for year in range(2086, 2101):
    inference_path = inference_dir+str(year)
    save_dir = inference_path+'/masks_psl_tmq'
    cgnet_load_create_masks(model_path, inference_path, save_dir, analyze=False, visualize=False)

100%|██████████| 730/730 [24:56<00:00,  2.05s/it]


  0%|          | 0/730 [00:00<?, ?it/s]

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2086/masks_psl_tmq


100%|██████████| 730/730 [24:06<00:00,  1.98s/it]


  0%|          | 0/730 [00:00<?, ?it/s]

...

100%|██████████| 730/730 [24:39<00:00,  2.03s/it]


  0%|          | 0/730 [00:00<?, ?it/s]

Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2099/masks_psl_tmq


100%|██████████| 730/730 [24:55<00:00,  2.05s/it]


Saved class masks to /glade/campaign/cgd/ccr/tking/cgnet/model_inference_data_from_katie/polar/sh/BRCP85C5CN/2100/masks_psl_tmq
CPU times: user 50min 8s, sys: 19min 41s, total: 1h 9min 50s
Wall time: 6h 12min 58s


## Pre-trained model which uses TMQ, V850, U850, and PSL

# See combine_class_masks.ipynb in order to generate heat maps